In [59]:
import pandas as pd
import re
import requests
from typing import List
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from opencc import OpenCC
cc = OpenCC('s2t')

In [ ]:
df = pd.read_csv("complaints_final.csv")
df.head()

In [ ]:
required_cols = [
    "Complaint ID",
    "Issue",
    "Sub-issue",
    "Consumer complaint narrative",
    "Consumer complaint narrative in Chinese",
    "Solution in Chinese"
]

df = df[required_cols].copy()
df = df.fillna("")
df.head()

In [62]:
stopwords = {
    "的", "了", "呢", "嗎", "我", "想", "請問", "一下", "如何", "怎麼", "要", "該", "是不是",
    "a", "an", "the", "is", "are", "to", "for", "and", "or", "in", "on", "with"
}

def normalize_text(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\u4e00-\u9fff\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text

def tokenize(text: str) -> List[str]:
    text = normalize_text(text)

    zh_blocks = re.findall(r"[\u4e00-\u9fff]+", text)
    en_tokens = re.findall(r"[a-zA-Z0-9_]+", text)

    zh_tokens = []
    for block in zh_blocks:
        block = block.strip()
        if len(block) == 1:
            zh_tokens.append(block)
        else:
            for n in [2, 3, 4]:
                for i in range(len(block) - n + 1):
                    zh_tokens.append(block[i:i+n])

    all_tokens = zh_tokens + en_tokens
    return [t for t in all_tokens if t not in stopwords and len(t) > 1]

In [63]:
def normalize_solution_text(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"\$[\d,]+(\.\d+)?", "金額", text)
    text = re.sub(r"\d+(\.\d+)?", "數字", text)
    text = re.sub(r"[^\w\u4e00-\u9fff\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def solution_similarity(sol1: str, sol2: str) -> float:
    tokens1 = set(tokenize(normalize_solution_text(sol1)))
    tokens2 = set(tokenize(normalize_solution_text(sol2)))

    if not tokens1 or not tokens2:
        return 0.0

    return len(tokens1 & tokens2) / max(len(tokens1), 1)

In [ ]:
def build_case_document(row: pd.Series) -> str:
    parts = [
        str(row.get("Issue", "")),
        str(row.get("Sub-issue", "")),
        str(row.get("Consumer complaint narrative in Chinese", "")),
        str(row.get("Consumer complaint narrative in Chinese", "")),
        str(row.get("Solution in Chinese", "")),
        str(row.get("Solution in Chinese", ""))
    ]
    return "\n".join([p for p in parts if p.strip() != ""])

df["case_document"] = df.apply(build_case_document, axis=1)
df[["Complaint ID", "case_document"]].head()

In [65]:
def keyword_overlap_score(question: str, case_doc: str) -> float:
    q_tokens = set(tokenize(question))
    c_tokens = set(tokenize(case_doc))

    if not q_tokens or not c_tokens:
        return 0.0

    return len(q_tokens & c_tokens) / max(len(q_tokens), 1)

In [66]:
def find_best_category(question: str, df: pd.DataFrame):
    category_df = (
        df[["Issue", "Sub-issue"]]
        .fillna("")
        .astype(str)
        .drop_duplicates()
        .reset_index(drop=True)
    )

    category_df["category_text"] = category_df["Issue"] + " | " + category_df["Sub-issue"]

    corpus = category_df["category_text"].tolist() + [question]

    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(2, 4),
        min_df=1
    )
    matrix = vectorizer.fit_transform(corpus)

    category_vectors = matrix[:-1]
    question_vector = matrix[-1]

    similarities = cosine_similarity(question_vector, category_vectors).flatten()
    best_idx = similarities.argmax()

    best_issue = category_df.loc[best_idx, "Issue"]
    best_sub_issue = category_df.loc[best_idx, "Sub-issue"]

    return best_issue, best_sub_issue

In [67]:
def rank_cases(question: str, df: pd.DataFrame, top_k: int = 3) -> pd.DataFrame:
    if "case_document" not in df.columns:
        raise ValueError("df 裡沒有 case_document 欄位，請先執行 build_case_document 那一格。")

    best_issue, best_sub_issue = find_best_category(question, df)

    filtered_df = df[
        (df["Issue"] == best_issue) &
        (df["Sub-issue"] == best_sub_issue)
    ].copy()

    # 如果篩完太少筆，退回用全部資料
    if len(filtered_df) < top_k:
        filtered_df = df.copy()

    docs = filtered_df["case_document"].tolist()
    corpus = docs + [question]

    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(2, 4),
        min_df=1
    )
    matrix = vectorizer.fit_transform(corpus)

    case_vectors = matrix[:-1]
    question_vector = matrix[-1]
    similarities = cosine_similarity(question_vector, case_vectors).flatten()

    rows = []
    for i, (_, row) in enumerate(filtered_df.iterrows()):
        sim = float(similarities[i])
        keyword_score = keyword_overlap_score(question, row["case_document"])
        final_score = (sim * 0.7) + (keyword_score * 0.3)

        rows.append({
            "Complaint ID": row.get("Complaint ID", ""),
            "Issue": row.get("Issue", ""),
            "Sub-issue": row.get("Sub-issue", ""),
            "Solution in Chinese": row.get("Solution in Chinese", ""),
            "similarity_score": sim,
            "keyword_score": keyword_score,
            "final_score": final_score
        })

    result_df = pd.DataFrame(rows).sort_values("final_score", ascending=False).reset_index(drop=True)

    unique_solutions = []

    for _, row in result_df.iterrows():
        current_solution = row["Solution in Chinese"]

        is_similar = False
        for saved_row in unique_solutions:
            sim_score = solution_similarity(current_solution, saved_row["Solution in Chinese"])
            if sim_score > 0.75:
                is_similar = True
                break

        if not is_similar:
            unique_solutions.append(row)

        if len(unique_solutions) >= top_k:
            break

    return pd.DataFrame(unique_solutions).reset_index(drop=True)

In [ ]:
best_issue, best_sub_issue = find_best_category(question, df)
print("Issue:", best_issue)
print("Sub-issue:", best_sub_issue)

In [33]:
question = "客戶帳戶被限制使用，而且無法提領資金，我應該先怎麼排查？"

In [ ]:
top_results = rank_cases(question, df, top_k=3)
top_results

In [ ]:
for i, row in top_results.iterrows():
    solution = str(row["Solution in Chinese"]).replace("建議處理方案（專業版）：", "").strip()

    print("=" * 80)
    print(f"建議方案 {i+1}")
    print(solution)
    print()

In [ ]:
# 簡體中文 ↔ 繁體中文 轉換 pip install opencc-python-reimplemented

Note: you may need to restart the kernel to use updated packages.


In [ ]:
#pip install ollama
'''
在終端機操作：
下載模型4.7G ollama pull qwen2.5:7b-instruct
執行 ollama pull qwen2.5:7b-instruct
查詢模型列表 ollama list
'''

In [70]:
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "qwen2.5:7b-instruct"

檢查 Ollama 服務有沒有啟動

In [71]:
#對網址發送 HTTP 請求
import requests

def check_ollama_service() -> bool:
    try:
        response = requests.get("http://localhost:11434/api/tags", timeout=10)
        # 11434 = Ollama 預設 port，/api/tags查看模型清單
        response.raise_for_status() #HTTP 狀態碼是不是正常
        return True
    except Exception as e:
        print("Ollama 服務連線失敗：", e)
        return False

service_ok = check_ollama_service()
print("Ollama 服務是否正常：", service_ok)

Ollama 服務是否正常： True


查看本機模型清單

In [ ]:
def list_ollama_models():
    try:
        response = requests.get("http://localhost:11434/api/tags", timeout=10)
        response.raise_for_status()
        data = response.json() #把 API 回傳的內容轉成 Python 字典
        models = data.get("models", []) #從 data 裡面取出 "models" 這個欄位，如果沒有，就給空 list []

        if not models:
            print("目前沒有找到任何模型。")
            return []

        model_names = [m.get("name", "") for m in models]
        #把每個模型物件裡的 "name" 取出來，整理成一個 list
        print("本機可用模型：")
        for name in model_names:
            print("-", name)
        return model_names

    except Exception as e:
        print("取得模型清單失敗：", e)
        return []

available_models = list_ollama_models()

本機可用模型：
- qwen2.5:7b-instruct


確認 qwen2.5:7b-instruct 是否存在

In [19]:
if OLLAMA_MODEL in available_models:
    print(f"模型已就緒：{OLLAMA_MODEL}")
else:
    print(f"找不到模型：{OLLAMA_MODEL}")
    print("請先在終端機執行：")
    print("ollama pull qwen2.5:7b-instruct")

模型已就緒：qwen2.5:7b-instruct


建立呼叫 Ollama 的函式

In [ ]:
def ask_ollama(prompt: str, model: str = OLLAMA_MODEL, timeout: int = 180) -> str:
    payload = {
        "model": model,
        "prompt": prompt,
        "system": "你是台灣的資深產品經理助理。你必須只使用繁體中文回答，禁止使用任何簡體字。",
        "stream": False #等模型整段回答好後，一次拿到完整結果。
    }

    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=timeout)
        response.raise_for_status()
        data = response.json()
        result = data.get("response", "").strip()
        result = cc.convert(result)   # 簡體轉繁體
        return result

    except requests.exceptions.ConnectionError:
        return "無法連線到 Ollama。請先確認 Ollama 應用程式或 serve 服務有啟動。"

    except requests.exceptions.Timeout:
        return "Ollama 回應逾時。"

    except requests.exceptions.HTTPError:
        return f"Ollama HTTP 錯誤：{response.text}"

    except Exception as e:
        return f"Ollama 呼叫失敗：{e}"

In [ ]:
#小測試
test_prompt = """
請只使用繁體中文回答，禁止使用任何簡體字。

問題：
客戶付款後沒有收到正確結果，PM 應該先檢查什麼？

請用 3 到 5 句話回答。
""".strip()

test_reply = ask_ollama(test_prompt)
print(test_reply)

首先，PM應確認付款流程是否無誤，包括金額及訂單資訊。其次，需檢查系統是否有正確生成對應的服務或產品資訊。再者，須與客服團隊核實是否有聯繫客戶並提供相關服務或產品。最後，應追溯是否有執行相關的後續步驟以確保客戶收到正確結果。


建立給 Ollama 的 prompt

In [ ]:
def build_prompt(question: str, top_results: pd.DataFrame) -> str:
    solution_blocks = []

    for i, row in top_results.iterrows():
        solution = str(row["Solution in Chinese"]).replace("建議處理方案（專業版）：", "").strip()
        solution_blocks.append(f"方案{i+1}：{solution}")

    solutions_text = "\n\n".join(solution_blocks)

    prompt = f"""
請只使用繁體中文回答，禁止使用任何簡體字。

你是台灣公司內部的資深 PM 助手，要幫一位沒有經驗的新手 PM 判斷問題。

使用者問題：
{question}

系統找到的參考方案：
{solutions_text}

請你根據上面的方案，整理成自然、白話、像真人主管帶新人的回答。

規則：
1. 先用 2 句話理解問題。
2. 再整理成 3 個排查方向。
3. 每個方向要講清楚「先做什麼」。
4. 只採用與問題主題一致的方案。
5. 如果某個方案明顯屬於別的題型，請直接忽略，不要寫進回答。
6. 若高相關方向不足三個，寧可只回答兩個，也不要硬湊不相關內容。
7. 對於「帳戶被限制、無法提領資金」這類問題，優先從：
   - 限制原因
   - 補件與審查進度
   - 資金返還或解除限制流程
   這三個方向整理。
8. 不要逐字抄原文，但要保留具體動作與關鍵詞，例如付款 ID、trace number、入帳紀錄、清算結果、沖回、補件、風控、合規、解除限制、返還餘額。
9. 請使用台灣常用繁體中文詞彙，例如「帳戶、後台、聯絡、資料、流程」，不要使用「賬戶、後臺、聯繫」這類中國用語。

最後請明確告訴我：第一步先查什麼，原因是什麼。
"""
    return prompt.strip()

In [25]:
#查看 prompt
prompt = build_prompt(question, top_results)
print(prompt[:3000])

你現在是一位公司內部的資深產品經理助理，正在協助一位完全沒有經驗的新手 PM。

請只使用繁體中文回答，禁止使用任何簡體字。

使用者的問題是：
消費者反映付款後沒有收到正確結果，該怎麼排查？

系統找出的三個可參考處理方案如下：
方案 1：
1. 先以 ACH / direct deposit trace 流程追查款項，確認付款方檔案是否成功送達、銀行是否收到入帳訊息，以及是否因例外處理而落入未分配帳。
2. 由後台作業單位比對付款 ID、trace number、批次檔與核心系統入帳紀錄，必要時直接與付款機關/雇主對接。
3. 若確認款項已送達銀行但未正確入帳，應立即將 該筆款項 補登至客戶帳戶，並退還透支費、退件費及其他連帶費用。
4. 若涉及政府補助、薪資或受保護給付，應提高優先級處理並保留合規紀錄，避免不當延誤。
5. 以正式書面回覆客戶：說明資金路徑、延遲原因、補帳結果與後續防呆措施。

方案 2：
1. 由支付作業單位核對每筆付款授權、取消回應碼、清算結果與帳務過帳紀錄，確認是否存在重複扣款或取消失敗。
2. 若系統已顯示交易失敗、取消或退回，應即刻沖回多扣資金，不得讓失敗交易持續佔用客戶餘額。
3. 檢視是否因此造成退票、租金/帳單延誤或其他連帶損失，並一併辦理費用沖銷或補償。
4. 如涉及第三方支付或外部商戶，應由銀行主動對接清算資訊，而非要求客戶自行追查。
5. 以書面通知客戶最終調整後之帳務明細、沖回金額與完成日。

方案 3：
1. 由存款作業單位比對兩家銀行的票據影像、重複存入標記、退票/沖正紀錄及實際清算結果，確認是否出現雙重扣回或重複沖銷。
2. 明確區分『客戶誤重複提交』與『銀行系統重複扣帳』之責任，避免在未釐清前直接以負餘額方式處理。
3. 若確認客戶未取得雙重利益，應立即恢復正確餘額、返還 $1100.00 及任何因錯誤扣帳產生之費用。
4. 如工資或薪資遭自動抵銷負餘額，應一併返還被占用資金。
5. 提供書面調查說明，載明票據流向、沖正原因、修正後帳務與完成日期。

請用繁體中文回答，並遵守以下規則：
1. 用像真人主管或資深 PM 帶新人的口吻回答，不要像機器。
2. 先簡短理解問題。
3. 再整理成 3 個可行排查方向。
4. 每個方向都要講得白話、實用。
5. 最後明確告訴使用者，如果資訊還不完整，應該先從哪個方向開始查。

In [37]:
#跑完整回答
final_reply = ask_ollama(prompt)
print(final_reply)

了解了，我們來看看怎樣排查消費者反映付款後沒有收到正確結果的問題吧。

首先，我們可以從以下三個方向來排查：

1. **確認付款流程**：先檢查付款是否確實成功送達銀行，包括 ACH 或直接存款等支付方式。看看是不是因為銀行或系統故障導致款項未入帳，或是被誤導至其他未分配的帳戶中。

2. **核對後台記錄**：和後勤部門一起，確認是否有付款 ID、Trace Number 等資訊與核心系統一致。如果有需要，直接和相關單位如政府部門或雇主對接以取得更多信息。這步驟可以幫助我們確定款項是否已達銀行但未正確入帳。

3. **客戶通知及補救措施**：如果確認款項未能如期入帳，我們應該立即將該筆款項補登至客戶帳戶中，并退還任何可能產生的費用如透支費和退件費等。對於涉及合規問題的付款，要特別提高處理優先級。

如果以上信息仍然不足以找出問題所在，建議首先從 **確認付款流程** 的方向開始查起。這樣可以確保我們不漏掉任何可能造成款項未入帳的細節，而且流程也是最直接且效率最高的方法。


In [74]:
#包成完整函式
def pm_ai_assistant(question: str, df: pd.DataFrame, top_k: int = 3) -> str:
    top_results = rank_cases(question, df, top_k=top_k)
    prompt = build_prompt(question, top_results)
    reply = ask_ollama(prompt)
    return reply

In [40]:
question = "消費者反映交易失敗，但帳戶還是被扣款，PM 應該先查什麼？"
answer = pm_ai_assistant(question, df, top_k=3)
print(answer)

1. 我理解這個問題是關於消費者反映交易失敗，但是帳戶還是被扣款的情況，我們需要先查清到底是哪些環節出了問題。
2. 排查方向一：確認支付流程。做什麼？首先要核對每筆付款授權、取消回應碼和清算結果，看看是否有重複扣款或取消失敗的狀況。
   排查方向二：查看帳務過帳紀錄。做什麼？檢查帳戶中的交易明細和入帳記錄，確認是否正確入帳。
   排查方向三：與銀行對接資訊。做什麼？如果涉及到第三方支付或外部商戶，應該由銀行主動提供清算資訊，確保所有信息都正確無誤。
3. 第一步先查付款流程，原因是這是最直接可以排查交易失敗原因的地方，從授權、取消到清算都能一一核對清楚。如果這裡沒有問題，再進一步查看帳務過帳紀錄；如果有可疑之處，則需要與銀行對接更多信息以確認是否有外部因素影響了交易結果。
4. 做什麼：先由支付作業單位核對每筆付款授權、取消回應碼和清算結果，確認是否存在重複扣款或取消失敗。


In [43]:
question = "消費者反映付款後沒有收到正確結果，該怎麼排查？"
answer = pm_ai_assistant(question, df, top_k=3)
print(answer)

1. 我理解這個問題就是消費者付了錢但是沒有收到正確的交易回饋，導致他可能很不滿意。我們要做的第一步是弄清楚到底是哪個環節出了問題，可能是銀行、系統或者我們自己的後台作業單位。

2. 排查方向一：檢查付款是否已成功到達銀行。
   做什麼：先用 ACH 或 direct deposit 的流程追蹤付款，確認付款方的檔案是否有成功送出、銀行是否收到入帳訊息，以及是否有因例外處理而被放置在未分配帳戶中。
   
   排查方向二：比對系統記錄與實際入帳情況。
   做什麼：由後台作業單位核對付款 ID、trace number 以及核心系統的入帳紀錄。必要時直接和付款機關或雇主對接。
   
   排查方向三：確認是否有其他問題導致資金未到達客戶帳戶。
   做什麼：如果發現款項已到銀行但未正確入帳，需立即補登至客戶帳戶，並退還可能產生的透支費、退件費及其他費用。如果是涉及政府補助或薪資給付，則要提高處理優先級。

3. 先查付款是否已經送到銀行（排查方向一），原因是這是問題的第一步，如果款項未到達銀行，那就不用再往下檢查了。

這樣一步一步來，我們就可以找到真正的原因並解決問題了。


In [41]:
question = "客戶表示帳戶開立失敗，懷疑是被錯誤標記風險，這種情況怎麼處理？"
answer = pm_ai_assistant(question, df, top_k=3)
print(answer)

1. 我理解這個問題是客戶開戶時遇到障礙，可能被錯誤標記為風險戶，我們需要確認並解決這項問題。

2. 排查方向一：重新檢查拒絕理由；方向二：收集客戶佐證資料；方向三：確保系統和流程無誤。

3. 做什麼：
   - 要做的是先由風控團隊重新檢核拒絕開戶原因，查看是否有錯誤標記的詐欺風險。
   - 接著向客戶索取報案紀錄、身分遭冒用聲明或其他相關文件，並進行人工覆核。
   - 最後確認系統和流程有沒有出現重複扣款或記錄錯誤的情況。

4. 先查拒絕開戶原因（方案 1 的第 1 步）。原因是客戶可能因為誤標而被拒絕開戶，我們需要確保這項判定是正確的。


In [79]:
question = "客戶帳戶被限制使用，而且無法提領資金，我應該先怎麼排查？"
answer = pm_ai_assistant(question, df, top_k=3)
print(answer)

好的，來幫忙排查這個問題吧！

首先，瞭解一下客戶賬戶被限制使用並無法提領資金的情況，這件事挺急的。接下來我們可以從三個方向來進行排查：

1. **確定限制原因**：第一步是先聯繫風控或合規團隊，確認客戶帳戶受限的原因。可能有身分驗證、可疑交易、退票風險等問題，這些都需要清楚瞭解。

2. **告知客戶需補件資料和時限**：第二步是即刻通知客戶需要補充哪些資料以及相關的驗證方式和截止時間。這樣可以避免客服與後臺信息不一致，讓客戶有明確的方向去處理。

3. **評估資金返還或解除限制**：第三步則是根據風控團隊提供的資訊，決定是否能先開放生活所需款項、加速退款或以正式支票方式返還。如果確定不再需要限制，應立即解除凍結並通知客戶。

這三步的順序是：第一步先查「確定限制原因」，原因是為了了解具體問題所在，才能後續進行對策規劃。
